# Exploring ChromaDB

Read what `ingest.py` stored in `./chroma_db`.

Run `python ingest.py` first so the collection has data. Select the `.venv` interpreter as the kernel.

In [1]:
import chromadb

client = chromadb.PersistentClient(path="chroma_db")

# What collections exist?
[c.name for c in client.list_collections()]

['documents']

In [2]:
collection = client.get_collection("documents")

print("chunks stored:", collection.count())

chunks stored: 9


## Read everything — `get()`

`get()` with no arguments returns every chunk: ids, text, and metadata.

In [3]:
data = collection.get()

# ChromaDB stores metadata as key-value rows internally, so dict key order
# is not preserved on read — sort the keys for a stable display.
for id_, meta in zip(data["ids"], data["metadatas"]):
    print(f"{id_:<18} {dict(sorted(meta.items()))}")

sample.html:0      {'chunk_index': 0, 'source': 'sample.html'}
sample.html:1      {'chunk_index': 1, 'source': 'sample.html'}
sample.html:2      {'chunk_index': 2, 'source': 'sample.html'}
sample.pdf:0       {'chunk_index': 0, 'page': 0, 'source': 'sample.pdf'}
sample.pdf:1       {'chunk_index': 1, 'page': 0, 'source': 'sample.pdf'}
sample.txt:0       {'chunk_index': 0, 'source': 'sample.txt'}
sample.txt:1       {'chunk_index': 1, 'source': 'sample.txt'}
sample.txt:2       {'chunk_index': 2, 'source': 'sample.txt'}
sample.txt:3       {'chunk_index': 3, 'source': 'sample.txt'}


## Read one chunk by id

In [4]:
one = collection.get(ids=["sample.pdf:0"])
print(one["documents"][0])

Object Storage and MinIO                                                        
                                                                                
MinIO is a high-performance, S3-compatible object storage server. In a RAG      
data pipeline it acts as the durable home for raw uploaded files: the upload    
service writes each incoming document to a MinIO bucket and immediately         
returns, while a background worker later fetches the object, extracts its


## Filter by metadata — `where`

Only chunks that came from a specific source file.

In [5]:
html_chunks = collection.get(where={"source": "sample.html"})
html_chunks["ids"]

['sample.html:0', 'sample.html:1', 'sample.html:2']

## Look at the embeddings

Embeddings are excluded by default (they're big). Ask for them with `include`.

In [6]:
with_vectors = collection.get(ids=["sample.txt:0"], include=["embeddings", "documents"])

vector = with_vectors["embeddings"][0]
print("embedding dimensions:", len(vector))
print("first 8 values:", [round(v, 4) for v in vector[:8]])

embedding dimensions: 384
first 8 values: [np.float64(-0.0919), np.float64(0.0268), np.float64(-0.0221), np.float64(0.0423), np.float64(-0.0639), np.float64(0.0693), np.float64(0.0149), np.float64(-0.0291)]


## Similarity search — `query()`

This is what retrieval does at question-answering time: embed the question, return the nearest chunks. Smaller distance = more similar.

In [7]:
results = collection.query(
    query_texts=["How does similarity search work?"],
    n_results=3,
)

for doc, meta, dist in zip(
    results["documents"][0], results["metadatas"][0], results["distances"][0]
):
    print(f"--- {meta['source']} chunk {meta['chunk_index']} (distance {dist:.4f})")
    print(doc[:200])
    print()

--- sample.html chunk 1 (distance 0.6948)
Similarity search

When a query arrives, the database embeds it and compares the query vector against stored vectors using a distance metric such as cosine distance or squared L2. The chunks with the 

--- sample.html chunk 2 (distance 1.0027)
Metadata filtering

Production systems usually attach metadata to each chunk, such as the source filename, page number, or tenant identifier. Filtering on metadata before or alongside the similarity s

--- sample.html chunk 0 (distance 1.1304)
Vector Databases Explained

A vector database stores high-dimensional embedding vectors and answers nearest-neighbor queries efficiently. ChromaDB is a popular open-source vector database that runs em



## Combine: similarity search + metadata filter

Search only within one document — how multi-tenant RAG systems scope queries per user.

In [8]:
results = collection.query(
    query_texts=["object storage"],
    n_results=2,
    where={"source": "sample.pdf"},
)

results["ids"][0]

['sample.pdf:0', 'sample.pdf:1']